# Algoritmo de Shor: Factorización Cuántica

**Objetivo:** entender el algoritmo de Shor y simular la estimación de período para N=15.

Shor factoriza N en O((log N)³) operaciones cuánticas, vs O(e^(N^(1/3))) del mejor algoritmo clásico.
La clave: encontrar el **período** r de la función f(x) = a^x mod N.

In [ ]:
import numpy as np
from math import gcd

# Parte clásica de Shor
def shor_classical(N, a):
    """Encuentra el período r tal que a^r ≡ 1 (mod N)"""
    x = 1
    for r in range(1, N+1):
        x = (x * a) % N
        if x == 1:
            return r
    return None

N, a = 15, 7
r = shor_classical(N, a)
print(f'a={a}, N={N}: período r = {r}')
print(f'Verificación: {a}^{r} mod {N} = {pow(a, r, N)}')

In [ ]:
# Factores a partir del período
def shor_factors(N, a, r):
    if r is None or r % 2 != 0:
        return None, None
    x = pow(a, r//2, N)
    if x == N-1:  # a^(r/2) ≡ -1 (mod N) — fallo
        return None, None
    p = gcd(x+1, N)
    q = gcd(x-1, N)
    return p, q

p, q = shor_factors(N, a, r)
print(f'Factores de {N}: {p} × {q} = {p*q}')

## La Parte Cuántica: QFT para Encontrar el Período

La QPE/QFT permite determinar r en O(log N) qubits de ancilla.
Idea: preparar Σ_x |x⟩|a^x mod N⟩ y aplicar QFT al primer registro.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

# Circuito de período finding para a=2, N=3 (caso mínimo)
# f(x) = 2^x mod 3: r=2 (0→1, 1→2, 2→1, ...)
# Usamos 2 qubits ancilla (2^2=4 > r=2)

def period_finding_2mod3():
    n_ancilla = 2
    n_work = 2
    qc = QuantumCircuit(n_ancilla + n_work)
    
    # Superposición en ancilla
    qc.h(range(n_ancilla))
    
    # Oráculo: codificación de f(x)=2^x mod 3
    # |00⟩ → f=1 → |01⟩
    # |01⟩ → f=2 → |10⟩
    # |10⟩ → f=1 → |01⟩ (período!)
    qc.cx(1, n_ancilla)        # control en q1
    qc.cx(0, n_ancilla+1)     # control en q0
    
    return qc

qc = period_finding_2mod3()
print(qc.draw())
print('\nEstado después del oráculo:')
sv = Statevector.from_instruction(qc)
print(sv.data.round(3))

In [ ]:
# Simulación completa: factorizar 15 con QPE
from qiskit.circuit.library import PhaseEstimation
from qiskit.quantum_info import Operator

# Para N=15, a=7: período r=4
# Operador U: |j⟩ → |7j mod 15⟩ — necesita 4 qubits trabajo

N_factor, a_factor = 15, 7
r_periodo = shor_classical(N_factor, a_factor)
print(f'N={N_factor}, a={a_factor}: r={r_periodo}')

# Factores
p, q = shor_factors(N_factor, a_factor, r_periodo)
print(f'Factores: {p} × {q}')

# Verificar otros valores de a
print('\nTodos los valores de a para N=15:')
for aa in range(2, N_factor):
    if gcd(aa, N_factor) == 1:  # coprimo
        rr = shor_classical(N_factor, aa)
        pp, qq = shor_factors(N_factor, aa, rr)
        exito = '✓' if pp and qq and pp*qq == N_factor else '✗'
        print(f'  a={aa:2d}: r={rr}, factores={pp}×{qq} {exito}')

## Complejidad y Escalado

In [ ]:
# Comparativa de complejidad
import matplotlib.pyplot as plt

bits = np.arange(4, 300)
N_vals = 2**bits

# Classical: mejor algoritmo GNFS ≈ exp(N^(1/3) * (log N)^(2/3) * 1.9)
gnafs = np.exp(1.9 * (bits * np.log(2))**(1/3) * (bits * np.log(2) * np.log(bits * np.log(2)))**(2/3))
# Quantum Shor: O((log N)^3)
shor = (bits)**3

plt.figure(figsize=(8,4))
plt.semilogy(bits, gnafs, 'r-', label='GNFS clásico')
plt.semilogy(bits, shor, 'b-', label='Shor cuántico')
plt.xlabel('Bits de N'); plt.ylabel('Operaciones (escala log)')
plt.title('Escalado de Shor vs GNFS')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('shor_scaling.png', dpi=80); plt.show()

## Próximos pasos

- **Lab:** `Cuadernos/laboratorios/10_qiskit_primitives_sampler_estimator.ipynb`
- **QPE completo:** `Cuadernos/laboratorios/07_phase_estimation_guiada.ipynb`
- **Siguiente guía:** `13_machine_learning_cuantico.ipynb`